# Geospatial Data

In [5]:
!pip cache purge --quiet

In [6]:
import pandas as pd

In [7]:
# If you've forked this repo, change OWNER to your GitHub username.
# REPO and BRANCH will normally stay the same unless you renamed them.
OWNER = "singlestore-cookbook"
REPO = "singlestore-cookbook.github.io"
BRANCH = "refs/heads/main"

BASE_URL = f"https://raw.githubusercontent.com/{OWNER}/{REPO}/{BRANCH}/code/part-multi-model/geospatial-data/datasets"

In [8]:
lines_csv_url = f"{BASE_URL}/london_lines.csv"

lines_df = pd.read_csv(lines_csv_url)

lines_df.columns = [col.lower().replace(" ", "_") for col in lines_df.columns]

In [9]:
lines_df.head()

,tube_line,color
0,Bakerloo,#B36305
1,Central,#E32017
2,Circle,#FFD300
3,District,#00782A
4,Hammersmith and City,#F3A9BB


In [10]:
connections_csv_url = f"{BASE_URL}/london_connections.csv"

connections_df = pd.read_csv(connections_csv_url)

connections_df.columns = [col.lower().replace(" ", "_") for col in connections_df.columns]

In [11]:
connections_df.head()

,tube_line,from_station,to_station
0,Bakerloo,Baker Street,Regents Park
1,Bakerloo,Charing Cross,Embankment
2,Bakerloo,Edgware Road (Bakerloo),Marylebone
3,Bakerloo,Embankment,Waterloo
4,Bakerloo,Harlesden,Willesden Junction


In [12]:
connections_df = connections_df[
    connections_df["tube_line"].isin(lines_df["tube_line"])
]

In [13]:
connections_df.head()

,tube_line,from_station,to_station
0,Bakerloo,Baker Street,Regents Park
1,Bakerloo,Charing Cross,Embankment
2,Bakerloo,Edgware Road (Bakerloo),Marylebone
3,Bakerloo,Embankment,Waterloo
4,Bakerloo,Harlesden,Willesden Junction


In [14]:
stations_csv_url = f"{BASE_URL}/london_stations.csv"

stations_df = pd.read_csv(stations_csv_url)

stations_df.columns = [col.lower().replace(" ", "_") for col in stations_df.columns]

In [15]:
stations_df.head()

,station,os_x,os_y,latitude,longitude,zone,postcode
0,Abbey Road,539081,183352,51.531952,0.003723,3,E15 3NB
1,Abbey Wood,547297,179002,51.490784,0.120272,4,SE2 9RH
2,Acton Central,520613,180299,51.508757,-0.263430,2,W3 6BH
3,Acton Main Line,520296,181196,51.516886,-0.267690,3,W3 9EH
4,Acton Town,519457,179639,51.503071,-0.280303,3,W3 8HN


In [16]:
stations_df = stations_df.drop(columns = ["os_x", "os_y", "postcode"])

In [17]:
# Get all unique stations appearing in connections_df, from both "from_station" and "to_station"
valid_stations = set(connections_df["from_station"]).union(set(connections_df["to_station"]))

# Filter stations_df to keep only rows where "station" is in valid_stations
stations_df = stations_df[stations_df["station"].isin(valid_stations)]

In [18]:
stations_df.head()

,station,latitude,longitude,zone
0,Abbey Road,51.531952,0.003723,3
4,Acton Town,51.503071,-0.280303,3
5,Addington Village,51.356239,-0.032665,"3,4,5,6"
6,Addiscombe,51.379808,-0.073213,"3,4,5,6"
8,Aldgate,51.514342,-0.075627,1


<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [19]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [20]:
tables = ["london_lines", "london_connections", "london_stations"]

with db_connection.begin() as conn:
    for table in tables:
        conn.execute(text(f"TRUNCATE TABLE {table};"))

In [21]:
lines_df.to_sql(
    "london_lines",
    con = db_connection,
    if_exists = "append",
    index = False,
    chunksize = 1000
)

13

In [22]:
connections_df.to_sql(
    "london_connections",
    con = db_connection,
    if_exists = "append",
    index = False,
    chunksize = 1000
)

458

In [23]:
stations_df.to_sql(
    "london_stations",
    con = db_connection,
    if_exists = "append",
    index = False,
    chunksize = 1000
)

349